# Project 4: CUAD Legal Contract Clause Multi-Label Classifier Fine-Tuning

**Halden Industrie GmbH** — Legal & Procurement Intelligence (Sofia Schmidt, `PL-WRO`)

This turnkey notebook fine-tunes a transformer model (`microsoft/deberta-v3-base` or `nlpaueb/legal-bert-base-uncased`) on the CUAD (Contract Understanding Atticus Dataset) multi-label clause benchmark.

- **Hardware**: Kaggle Free GPU (Tesla T4 or P100, $0 card-free tier)
- **Task**: Multi-Label Binary Cross-Entropy Loss with Logits
- **Target Metric**: Multi-label Macro-F1 $\ge 0.85$, Span Grounding Accuracy $\ge 95\%$
- **Export**: HuggingFace & ONNX lightweight artifacts for local CPU deployment


In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets accelerate evaluate scikit-learn onnx onnxruntime


In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Step 2: Define CUAD Categories
CUAD_CATEGORIES = [
    "Limitation of Liability",
    "Cap on Liability",
    "Indemnification",
    "Governing Law",
    "Jurisdiction",
    "Termination for Convenience",
    "Notice Period To Terminate",
    "Non-Compete",
    "Exclusivity",
    "Confidentiality",
    "IP Assignment",
    "Warranty",
    "Audit Rights",
    "Force Majeure",
    "Insurance",
    "Price and Payment Terms",
    "Renewal Term"
]
num_labels = len(CUAD_CATEGORIES)
label2id = {label: i for i, label in enumerate(CUAD_CATEGORIES)}
id2label = dict(enumerate(CUAD_CATEGORIES))


In [ ]:
# Step 3: Load Model and Tokenizer
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
).to(device)


In [ ]:
# Step 4: Metric computation for Multi-Label F1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid
    preds = (probs >= 0.50).astype(int)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    return {"macro_f1": macro_f1, "micro_f1": micro_f1}


In [ ]:
# Step 5: Training Arguments (Optimized for 16GB GPU)
training_args = TrainingArguments(
    output_dir="./cuad_output",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50
)


In [ ]:
# Step 6: Export Model Artifacts for Local Deployment
EXPORT_PATH = "/kaggle/working/cuad_contract_clf"
print(f"Exporting trained model to {EXPORT_PATH}...")
# model.save_pretrained(EXPORT_PATH)
# tokenizer.save_pretrained(EXPORT_PATH)
